In [ ]:
import sys

print(sys.executable)

In [ ]:
from pathlib import Path

# Move one level up to the project root
project_path = Path.cwd().parent

print("Project directory:", project_path)

# Locate the raw data folder
raw_path = project_path / "data" / "raw"

print("\nContents of data/raw:")

for item in raw_path.iterdir():
    print(item.name, "(Folder)" if item.is_dir() else "(File)")

In [ ]:
# Define the dataset directory
dataset_path = raw_path / "TOLIFE_COPD_Dataset"

# Display immediate contents
print("DATASET STRUCTURE\n")

for item in sorted(dataset_path.iterdir()):
    if item.is_dir():
        print(f"[FOLDER] {item.name}")
    else:
        print(f"[FILE]   {item.name}")

# Count all CSV files recursively
csv_files = list(dataset_path.rglob("*.csv"))

print("\nTotal CSV files:", len(csv_files))

print("\nFirst 15 CSV file paths:")
for file in csv_files[:15]:
    print(file.relative_to(dataset_path))

In [ ]:

from collections import Counter

# Identify participant directories
participant_dirs = [
    folder
    for country_folder in (dataset_path / "index").iterdir()
    if country_folder.is_dir()
    for folder in country_folder.iterdir()
    if folder.is_dir()
]

# Count participants by country
country_counts = Counter(
    folder.parent.name for folder in participant_dirs
)

print("PARTICIPANT INVENTORY")
print("---------------------")
print("Total participants:", len(participant_dirs))

for country, count in sorted(country_counts.items()):
    print(f"{country}: {count}")

# Count CSV files by filename
file_counts = Counter(
    file.name for file in dataset_path.rglob("*.csv")
)

print("\nCSV FILE INVENTORY")
print("------------------")

for filename, count in sorted(file_counts.items()):
    print(f"{filename}: {count}")

# Count participants with exacerbation records
participants_with_exacerbations = sum(
    1 for folder in participant_dirs
    if (folder / "exacerbations.csv").exists()
)

print("\nEXACERBATION RECORDS")
print("--------------------")
print(
    "Participants with exacerbation files:",
    participants_with_exacerbations
)


In [ ]:
import pandas as pd

print("Pandas version:", pd.__version__)

In [ ]:
# Select a participant folder
sample_folder = dataset_path / "index" / "GERMANY" / "GER-001"

# Inspect every CSV file belonging to this participant
for file in sorted(sample_folder.glob("*.csv")):

    df = pd.read_csv(file)

    print("\n" + "=" * 60)
    print("FILE:", file.name)
    print("=" * 60)

    print("Shape:", df.shape)

    print("\nColumns:")
    print(df.columns.tolist())

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nFirst 3 rows:")
    display(df.head(3))

In [ ]:

import pandas as pd
from pathlib import Path

# Sensor files to investigate
sensor_files = [
    "BRI.csv",
    "EI.csv",
    "MI.csv",
    "PEI.csv",
    "PRVI.csv",
    "SHRI.csv",
    "SII.csv",
    "SQI.csv"
]

records = []

# Go through Germany and Spain
for country in ["GERMANY", "SPAIN"]:

    country_path = dataset_path / "index" / country

    for participant_folder in country_path.iterdir():

        if not participant_folder.is_dir():
            continue

        participant_id = participant_folder.name

        for filename in sensor_files:

            file_path = participant_folder / filename

            if not file_path.exists():
                continue

            df = pd.read_csv(file_path)

            for flag in ["index_validity", "data_validity"]:

                if flag not in df.columns:
                    continue

                counts = df[flag].value_counts(dropna=False)

                for value, count in counts.items():

                    records.append({
                        "participant": participant_id,
                        "country": country,
                        "sensor": filename.replace(".csv", ""),
                        "flag": flag,
                        "flag_value": value,
                        "row_count": count,
                        "total_rows": len(df)
                    })

# Create detailed report
validity_report = pd.DataFrame(records)

# Aggregate across participants
summary = (
    validity_report
    .groupby(["sensor", "flag", "flag_value"], dropna=False)
    .agg(
        total_rows=("row_count", "sum"),
        participants=("participant", "nunique")
    )
    .reset_index()
)

# Calculate percentage within each sensor and flag
summary["percentage"] = (
    summary["total_rows"] /
    summary.groupby(["sensor", "flag"])["total_rows"].transform("sum")
) * 100

summary["percentage"] = summary["percentage"].round(2)

summary = summary.sort_values(
    ["sensor", "flag", "flag_value"],
    na_position="last"
)

display(summary)


In [ ]:

cross_tab_results = []

for country in ["GERMANY", "SPAIN"]:

    country_path = dataset_path / "index" / country

    for participant_folder in country_path.iterdir():

        if not participant_folder.is_dir():
            continue

        for filename in sensor_files:

            file_path = participant_folder / filename

            if not file_path.exists():
                continue

            df = pd.read_csv(file_path)

            if not {
                "index_validity",
                "data_validity"
            }.issubset(df.columns):
                continue

            counts = (
                df.groupby(
                    ["data_validity", "index_validity"],
                    dropna=False
                )
                .size()
                .reset_index(name="row_count")
            )

            for _, row in counts.iterrows():

                cross_tab_results.append({
                    "sensor": filename.replace(".csv", ""),
                    "data_validity": row["data_validity"],
                    "index_validity": row["index_validity"],
                    "row_count": row["row_count"]
                })

cross_tab = pd.DataFrame(cross_tab_results)

cross_tab_summary = (
    cross_tab
    .groupby(
        ["sensor", "data_validity", "index_validity"],
        dropna=False
    )["row_count"]
    .sum()
    .reset_index()
)

display(cross_tab_summary)


In [ ]:

zero_audit_records = []

for country in ["GERMANY", "SPAIN"]:

    country_path = dataset_path / "index" / country

    for participant_folder in country_path.iterdir():

        if not participant_folder.is_dir():
            continue

        for filename in sensor_files:

            file_path = participant_folder / filename

            if not file_path.exists():
                continue

            df = pd.read_csv(file_path)

            measurement_cols = [
                col for col in df.select_dtypes(include="number").columns
                if col not in ["index_validity", "data_validity"]
            ]

            if not measurement_cols:
                continue

            measurements = df[measurement_cols]

            non_missing = measurements.notna()

            all_zero = (
                measurements.eq(0).where(non_missing, True).all(axis=1)
                & non_missing.any(axis=1)
            )

            index_flag = df["index_validity"]

            zero_audit_records.append({
                "sensor": filename.replace(".csv", ""),
                "total_rows": len(df),
                "all_zero_rows": int(all_zero.sum()),
                "index_0": int((all_zero & index_flag.eq(0)).sum()),
                "index_1": int((all_zero & index_flag.eq(1)).sum()),
                "index_missing": int((all_zero & index_flag.isna()).sum()),
                "rows_with_missing_measurements": int(
                    measurements.isna().any(axis=1).sum()
                )
            })

zero_audit = pd.DataFrame(zero_audit_records)

zero_summary = (
    zero_audit
    .groupby("sensor")
    .sum(numeric_only=True)
    .reset_index()
)

zero_summary["all_zero_percentage"] = (
    zero_summary["all_zero_rows"] /
    zero_summary["total_rows"] * 100
).round(2)

display(zero_summary)



# Stage 2: Dataset-Wide Data Quality Assessment

## 2.1 Validity Flag Investigation

### Objective
Understand the two validity indicators available in the sensor/index CSV files before making any data-cleaning decisions.

### Official Definitions (Dataset README)

According to the TOLIFE dataset README:

- `data_validity = 1`: Raw sensor data was available.
- `data_validity = 0`: Raw sensor data was missing.
- `index_validity = 1`: Calculated health index was valid.
- `index_validity = 0`: Calculated health index was invalid.

### Analysis Performed

Audited the distribution of both validity flags across all available participant sensor files.

Also examined the combinations of `data_validity` and `index_validity` to understand their relationship.

### Key Findings

1. All seven streams with both flags present contain rows where both flags are 0.
2. Seven streams contain rows where `data_validity = 1` but `index_validity = 0`.
3. No rows were observed with `data_validity = 0` and `index_validity = 1`.
4. PRVI contains 505 rows with missing `index_validity` values.
5. PRVI has 2,917 rows with `data_validity = 1`.

### Interpretation

The two flags represent different aspects of data quality.

Raw sensor data may be available even when the calculated health index is invalid.

Therefore, the flags must be evaluated separately during preprocessing.

### Cleaning Decision

No records have been deleted or modified.

Further investigation is required before deciding how to handle invalid indices and zero-valued measurements.

---

## 2.2 Zero-Value Investigation

### Objective

Investigate rows in which all available numerical health measurements are zero.

### Planned Analysis

- Count all-zero measurement rows for each sensor stream.
- Compare these rows against `index_validity`.
- Identify rows with missing measurement values.
- Investigate whether zero-valued records are associated with invalid health indices.

### Cleaning Decision

No zero-valued records will be removed or replaced until their meaning has been investigated.


In [ ]:

missing_records = []

for country in ["GERMANY", "SPAIN"]:

    country_path = dataset_path / "index" / country

    for participant_folder in country_path.iterdir():

        if not participant_folder.is_dir():
            continue

        participant_id = participant_folder.name

        for filename in sensor_files:

            file_path = participant_folder / filename

            if not file_path.exists():
                continue

            df = pd.read_csv(file_path)
            sensor = filename.replace(".csv", "")

            for column in df.columns:

                missing_records.append({
                    "participant": participant_id,
                    "sensor": sensor,
                    "column": column,
                    "total_values": len(df),
                    "missing_values": int(df[column].isna().sum())
                })


# Convert results into a DataFrame
missing_details = pd.DataFrame(missing_records)

# Calculate total and missing values for each sensor column
missing_summary = (
    missing_details
    .groupby(["sensor", "column"])
    .agg(
        total_values=("total_values", "sum"),
        missing_values=("missing_values", "sum")
    )
    .reset_index()
)

# Count how many participants have missing values in each column
affected_participants = (
    missing_details[missing_details["missing_values"] > 0]
    .groupby(["sensor", "column"])["participant"]
    .nunique()
    .reset_index(name="participants_affected")
)

# Combine both summaries
missing_summary = missing_summary.merge(
    affected_participants,
    on=["sensor", "column"],
    how="left"
)

missing_summary["participants_affected"] = (
    missing_summary["participants_affected"].fillna(0).astype(int)
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_values"] /
    missing_summary["total_values"] * 100
).round(2)

# Display only columns containing missing values
missing_summary = (
    missing_summary[missing_summary["missing_values"] > 0]
    .sort_values(
        ["sensor", "missing_percentage"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

display(missing_summary)


### Dataset-Wide Missing-Value Audit

**Objective:** Identify missing measurements across all available participant sensor files and determine which variables and participants are affected.

**Method:**
- Iterated through all available participant folders in Germany and Spain.
- Inspected every column in the eight sensor streams.
- Counted total values and missing values for each sensor-column combination.
- Calculated missing percentages and the number of participants affected.

**Observed results:**

1. **BRI:** Three measurement columns (`rf_max`, `rf_min`, `rf_std`) each had one missing value.
2. **PEI:** `HR` and `SpO2` each had 427 missing values (4.31%). `mean_FEV1` and `mean_PEF` each had 58 missing values (0.59%).
3. **PRVI:** Missingness was relatively high. `RMSSD`, `mean_HR`, and `std_RR` each had 505 missing values (17.31%). `nRMSSD` had 672 missing values (23.08%). Its range columns had up to 28.89% missingness.
4. **SQI:** `sleep_duration` had 10 missing values (0.10%), and `SE` had 4 missing values (0.04%). The undocumented columns `use_time` and `duration_movements` each had 433 missing values out of 437 recorded values (99.08%).

**Interpretation:**
- PRVI has substantial missingness across several cardiovascular measurements.
- PEI missingness is concentrated in `HR` and `SpO2`.
- BRI has negligible missingness.
- The two undocumented SQI columns have extremely high missingness and may not be suitable for modelling.
- The denominators for some columns differ from the total sensor-row count. This suggests that columns may not be consistently present across participant files.

**Important distinction:** This audit counts missing cells in columns that exist. It does not count entirely absent columns or missing sensor files.

**Decision:** No missing values were imputed, and no columns or records were removed. A schema-consistency audit is required before cleaning.

In [ ]:

schema_records = []

for country in ["GERMANY", "SPAIN"]:

    country_path = dataset_path / "index" / country

    for participant_folder in country_path.iterdir():

        if not participant_folder.is_dir():
            continue

        participant_id = participant_folder.name

        for filename in sensor_files:

            file_path = participant_folder / filename

            if not file_path.exists():
                continue

            df = pd.read_csv(file_path)

            schema_records.append({
                "participant": participant_id,
                "sensor": filename.replace(".csv", ""),
                "columns": tuple(sorted(df.columns)),
                "column_count": len(df.columns)
            })


schema_df = pd.DataFrame(schema_records)

# Count how many participants have each unique schema
schema_summary = (
    schema_df
    .groupby(["sensor", "columns"])
    .agg(
        participants=("participant", "nunique"),
        column_count=("column_count", "first")
    )
    .reset_index()
)

display(schema_summary)


In [ ]:

# Find the most common column structure for each sensor
expected_schemas = (
    schema_df.groupby("sensor")["columns"]
    .agg(lambda x: x.value_counts().index[0])
    .to_dict()
)

schema_differences = []

for _, row in schema_df.iterrows():

    expected = set(expected_schemas[row["sensor"]])
    actual = set(row["columns"])

    missing_columns = sorted(expected - actual)
    extra_columns = sorted(actual - expected)

    if missing_columns or extra_columns:

        schema_differences.append({
            "participant": row["participant"],
            "sensor": row["sensor"],
            "missing_columns": missing_columns,
            "extra_columns": extra_columns
        })

schema_differences_df = pd.DataFrame(schema_differences)

display(schema_differences_df)


### Sensor Schema Consistency Audit

**Objective:** Determine whether sensor CSV files have consistent column structures across all participants.

**Method:**
- Inspected the columns of every available participant sensor file.
- Grouped files by sensor and unique column structure.
- Counted how many participants shared each structure.

**Observed results:**
- BRI, EI, MI, PEI, SHRI and SII had identical column structures across all 74 participants.
- PRVI had two different structures: 63 files contained 15 columns, while 2 contained 12 columns. Nine participants had no PRVI file.
- SQI had two different structures: 72 files contained 7 columns, while 2 contained 9 columns.

**Interpretation:**
Most sensor streams have consistent schemas. PRVI and SQI have structural differences that require further investigation. These differences may affect dataset integration and feature availability.

**Decision:** No columns were added, removed, or renamed. The exact participants and differing columns will be identified before deciding how to standardize the data.

### Schema Consistency Audit — Detailed Findings

**Objective:** Identify the exact participants and columns responsible for differences in sensor-file structures.

**Observed results:**

Four participant files differed from their sensor's most common schema:

| Participant | Sensor | Missing columns | Extra columns |
|---|---|---|---|
| GER-001 | SQI | None | `duration_movements`, `use_time` |
| GER-004 | SQI | None | `duration_movements`, `use_time` |
| SPA-005 | PRVI | `nRMSSD`, `range_nRMSSD_high`, `range_nRMSSD_low` | None |
| SPA-029 | PRVI | `nRMSSD`, `range_nRMSSD_high`, `range_nRMSSD_low` | None |

**Interpretation:**
- The two additional SQI columns occur only in GER-001 and GER-004. Both were also found to have approximately 99.08% missing values across their recorded entries.
- SPA-005 and SPA-029 lack three nRMSSD-related PRVI variables.
- These differences represent structural inconsistencies, not necessarily erroneous records.

**Decision:**
No original columns or files were modified. During later dataset integration, we may standardize the schemas by introducing absent columns as missing values, while retaining the original raw data.



In [ ]:

duplicate_records = []

for country in ["GERMANY", "SPAIN"]:

    country_path = dataset_path / "index" / country

    for participant_folder in country_path.iterdir():

        if not participant_folder.is_dir():
            continue

        participant_id = participant_folder.name

        for filename in sensor_files:

            file_path = participant_folder / filename

            if not file_path.exists():
                continue

            df = pd.read_csv(file_path)

            # Convert timestamps to UTC datetime
            timestamps = pd.to_datetime(
                df["Timestamp"],
                utc=True,
                errors="coerce"
            )

            # Identify duplicate timestamps
            valid_timestamps = timestamps.dropna()

            duplicate_mask = (
                timestamps.notna()
                & timestamps.duplicated(keep="first")
            )

            duplicate_groups = (
                valid_timestamps.value_counts()
                .gt(1)
                .sum()
            )

            duplicate_records.append({
                "participant": participant_id,
                "sensor": filename.replace(".csv", ""),
                "total_rows": len(df),
                "missing_timestamps": int(timestamps.isna().sum()),
                "duplicate_timestamp_groups": int(duplicate_groups),
                "duplicate_extra_rows": int(duplicate_mask.sum()),
                "rows_in_duplicate_groups": int(
                    timestamps.duplicated(keep=False).sum()
                )
            })


duplicate_details = pd.DataFrame(duplicate_records)

# Summarize results by sensor
duplicate_summary = (
    duplicate_details
    .groupby("sensor")
    .agg(
        total_rows=("total_rows", "sum"),
        missing_timestamps=("missing_timestamps", "sum"),
        duplicate_timestamp_groups=(
            "duplicate_timestamp_groups", "sum"
        ),
        duplicate_extra_rows=("duplicate_extra_rows", "sum"),
        rows_in_duplicate_groups=(
            "rows_in_duplicate_groups", "sum"
        ),
        participants_affected=(
            "duplicate_extra_rows",
            lambda x: (x > 0).sum()
        )
    )
    .reset_index()
)

display(duplicate_summary)


### Duplicate Timestamp Audit

**Objective:** Identify repeated timestamps within individual participant sensor files.

**Method:**
- Inspected all available files across the eight sensor streams.
- Converted timestamps to UTC datetime format.
- Identified timestamp values occurring more than once within each participant's sensor file.
- Counted duplicate timestamp groups, additional occurrences, and affected participants.
- Checked for missing or unparseable timestamps.

**Observed results:**

| Sensor | Duplicate timestamp groups | Additional rows | Participants affected |
|---|---:|---:|---:|
| BRI | 0 | 0 | 0 |
| EI | 70 | 70 | 35 |
| MI | 0 | 0 | 0 |
| PEI | 0 | 0 | 0 |
| PRVI | 195 | 195 | 34 |
| SHRI | 0 | 0 | 0 |
| SII | 70 | 70 | 35 |
| SQI | 0 | 0 | 0 |

No missing or unparseable timestamps were observed in the audited sensor files.

**Interpretation:**
- Duplicate timestamps occur in EI, PRVI and SII.
- EI and SII have matching duplicate counts, which warrants further investigation.
- PRVI has the highest number of duplicate timestamp groups.
- A repeated timestamp does not necessarily mean that the entire record is duplicated. Measurements associated with the same timestamp may differ.

**Decision:** No records were removed. A further audit is required to distinguish identical duplicate records from records with conflicting measurements at the same timestamp.

In [ ]:

duplicate_type_records = []

for country in ["GERMANY", "SPAIN"]:

    country_path = dataset_path / "index" / country

    for participant_folder in country_path.iterdir():

        if not participant_folder.is_dir():
            continue

        participant_id = participant_folder.name

        for filename in sensor_files:

            file_path = participant_folder / filename

            if not file_path.exists():
                continue

            df = pd.read_csv(file_path)

            timestamps = pd.to_datetime(
                df["Timestamp"],
                utc=True,
                errors="coerce"
            )

            temp = df.copy()
            temp["_parsed_timestamp"] = timestamps

            duplicate_groups = temp[
                temp["_parsed_timestamp"].notna()
            ].groupby("_parsed_timestamp")

            for timestamp, group in duplicate_groups:

                if len(group) <= 1:
                    continue

                # Compare all original columns except Timestamp
                measurement_data = group.drop(
                    columns=["Timestamp", "_parsed_timestamp"]
                )

                unique_records = measurement_data.drop_duplicates()

                if len(unique_records) == 1:
                    duplicate_type = "identical"
                else:
                    duplicate_type = "conflicting"

                duplicate_type_records.append({
                    "participant": participant_id,
                    "sensor": filename.replace(".csv", ""),
                    "timestamp": timestamp,
                    "duplicate_count": len(group),
                    "duplicate_type": duplicate_type
                })


duplicate_types_df = pd.DataFrame(duplicate_type_records)

# Summarize duplicate types by sensor
duplicate_type_summary = (
    duplicate_types_df
    .groupby(["sensor", "duplicate_type"])
    .agg(
        duplicate_groups=("timestamp", "count"),
        participants_affected=("participant", "nunique")
    )
    .reset_index()
)

display(duplicate_type_summary)


### Duplicate Record Classification

**Objective:** Determine whether records sharing the same timestamp contain identical or conflicting data.

**Method:**
- Grouped records by participant, sensor and timestamp.
- Compared all original columns except `Timestamp` within each duplicate group.
- Classified groups as identical when all compared values matched, and conflicting when at least one value differed.

**Observed results:**

| Sensor | Identical groups | Conflicting groups | Total groups |
|---|---:|---:|---:|
| EI | 56 | 14 | 70 |
| PRVI | 26 | 169 | 195 |
| SII | 58 | 12 | 70 |

**Interpretation:**
- EI and SII contain predominantly identical duplicate records.
- PRVI contains predominantly conflicting duplicate records.
- Conflicting records may represent different measurements recorded at the same timestamp, so they cannot automatically be considered redundant.
- Identical records may be candidates for deduplication after verification.

**Decision:**
No records were deleted or modified. Conflicting timestamp groups will be inspected before selecting a resolution strategy. Raw data will remain unchanged.

In [ ]:

conflicting_samples = []

for country in ["GERMANY", "SPAIN"]:

    country_path = dataset_path / "index" / country

    for participant_folder in country_path.iterdir():

        if not participant_folder.is_dir():
            continue

        participant_id = participant_folder.name

        for filename in sensor_files:

            file_path = participant_folder / filename

            if not file_path.exists():
                continue

            sensor = filename.replace(".csv", "")

            if sensor not in ["EI", "PRVI", "SII"]:
                continue

            df = pd.read_csv(file_path)

            timestamps = pd.to_datetime(
                df["Timestamp"],
                utc=True,
                errors="coerce"
            )

            temp = df.copy()
            temp["_parsed_timestamp"] = timestamps

            groups = temp[
                temp["_parsed_timestamp"].notna()
            ].groupby("_parsed_timestamp")

            for timestamp, group in groups:

                if len(group) <= 1:
                    continue

                comparison = group.drop(
                    columns=["Timestamp", "_parsed_timestamp"]
                )

                if len(comparison.drop_duplicates()) > 1:

                    conflicting_samples.append({
                        "participant": participant_id,
                        "sensor": sensor,
                        "timestamp": timestamp,
                        "records": group.drop(
                            columns=["_parsed_timestamp"]
                        )
                    })


# Display up to three examples per sensor
for sensor in ["EI", "PRVI", "SII"]:

    samples = [
        item for item in conflicting_samples
        if item["sensor"] == sensor
    ][:3]

    print(f"\n{'=' * 60}")
    print(f"SENSOR: {sensor}")
    print(f"{'=' * 60}")

    for sample in samples:

        print(
            f"\nParticipant: {sample['participant']}"
            f"\nTimestamp: {sample['timestamp']}"
        )

        display(sample["records"])


### Investigation of Conflicting Duplicate Timestamps

**Objective:** Inspect records sharing timestamps to understand which variables differ and whether differences involve measurements or validity flags.

**Observed results:**

**EI:**
- Inspected examples from GER-004 and GER-007.
- Environmental measurement values were identical between records sharing a timestamp.
- `data_validity` differed between 0 and 1.
- `index_validity` remained 0 in the inspected examples.

**SII:**
- Inspected examples from GER-004 and GER-007.
- Sound-intensity measurements were identical between records sharing a timestamp.
- `data_validity` differed between 0 and 1.
- `index_validity` remained 0 in the inspected examples.

**PRVI:**
- Inspected three conflicting timestamp groups from GER-003.
- Actual measurements such as `mean_HR`, `std_RR`, `RMSSD` and `nRMSSD` differed slightly between records.
- Several range-related columns were populated in one record but missing in the other.
- Both records had `data_validity = 1` and `index_validity = 1` in the inspected examples.

**Interpretation:**
- EI and SII examples suggest measurement-identical records with conflicting data-availability flags.
- PRVI examples show differences in actual calculated measurements, not just validity metadata.
- The reason for multiple PRVI records sharing a timestamp is not established and requires further investigation.

**Decision:**
No duplicate records were removed or modified. EI and SII flag differences and PRVI measurement differences will be treated separately when designing the cleaning strategy.

In [ ]:

column_difference_records = []

for country in ["GERMANY", "SPAIN"]:

    country_path = dataset_path / "index" / country

    for participant_folder in country_path.iterdir():

        if not participant_folder.is_dir():
            continue

        participant_id = participant_folder.name

        for filename in sensor_files:

            file_path = participant_folder / filename

            if not file_path.exists():
                continue

            sensor = filename.replace(".csv", "")

            if sensor not in ["EI", "PRVI", "SII"]:
                continue

            df = pd.read_csv(file_path)

            timestamps = pd.to_datetime(
                df["Timestamp"],
                utc=True,
                errors="coerce"
            )

            temp = df.copy()
            temp["_parsed_timestamp"] = timestamps

            groups = temp[
                temp["_parsed_timestamp"].notna()
            ].groupby("_parsed_timestamp")

            for timestamp, group in groups:

                if len(group) <= 1:
                    continue

                comparison = group.drop(
                    columns=["Timestamp", "_parsed_timestamp"]
                )

                # Find columns with more than one distinct value
                differing_columns = comparison.columns[
                    comparison.nunique(dropna=False) > 1
                ]

                for column in differing_columns:

                    column_difference_records.append({
                        "participant": participant_id,
                        "sensor": sensor,
                        "timestamp": timestamp,
                        "column": column
                    })


column_differences = pd.DataFrame(
    column_difference_records
)

column_difference_summary = (
    column_differences
    .groupby(["sensor", "column"])
    .agg(
        conflicting_groups=("timestamp", "count"),
        participants_affected=("participant", "nunique")
    )
    .reset_index()
    .sort_values(
        ["sensor", "conflicting_groups"],
        ascending=[True, False]
    )
)

display(column_difference_summary)



### Duplicate Timestamp Difference Investigation

**Objective:** Identify which columns differ between records that share the same timestamp, so that duplicate records can be classified more precisely before any cleaning decisions are made.

**Method:**
- Grouped records by participant, sensor, and parsed timestamp.
- Examined duplicate timestamp groups only.
- Compared every column except the original timestamp.
- Counted the number of duplicate groups and participants affected for each differing column.
- Treated missing values as distinct from populated values during comparison.

**Observed results:**

| Sensor | Differing columns | Duplicate groups | Participants affected |
|---|---|---:|---:|
| EI | `data_validity` | 14 | 7 |
| SII | `data_validity` | 12 | 6 |
| PRVI | `mean_HR`, `std_RR`, `RMSSD`, `nRMSSD`, and associated range columns | 169* | 27 |

\*The two `range_nRMSSD` columns differed in 164 groups; the other listed PRVI columns differed in 169 groups.

**Interpretation:**

- **EI and SII:** The only differing column was `data_validity`. These groups are measurement-identical duplicates with conflicting validity flags.
- **PRVI:** Differences extend to actual measurement and range columns. These are measurement-level conflicts, not merely validity-flag differences.
- The reason for the PRVI differences cannot be established from this audit alone.

**Cleaning decision:** No duplicate records were deleted or modified. EI/SII validity-flag conflicts and PRVI measurement-level conflicts will be retained for further consideration before defining a cleaning policy.

**Limitation:** This investigation identifies which values differ, but does not establish why duplicate timestamps occur or which record, if any, should be retained.


In [ ]:

sensor_names = ["BRI", "EI", "MI", "PEI", "PRVI", "SHRI", "SII", "SQI"]

coverage_records = []

for country in ["GERMANY", "SPAIN"]:

    country_path = dataset_path / "index" / country

    for participant_folder in country_path.iterdir():

        if not participant_folder.is_dir():
            continue

        participant_id = participant_folder.name

        for sensor in sensor_names:

            file_path = participant_folder / f"{sensor}.csv"

            if not file_path.exists():
                continue

            df = pd.read_csv(file_path)

            timestamps = pd.to_datetime(
                df["Timestamp"],
                utc=True,
                errors="coerce"
            ).dropna()

            unique_timestamps = (
                timestamps
                .drop_duplicates()
                .sort_values()
            )

            # Calculate intervals between consecutive unique timestamps
            intervals = unique_timestamps.diff().dropna()

            median_interval_hours = (
                intervals.median().total_seconds() / 3600
                if not intervals.empty
                else None
            )

            coverage_records.append({
                "participant": participant_id,
                "sensor": sensor,
                "total_rows": len(df),
                "unique_timestamps": len(unique_timestamps),
                "earliest_timestamp": unique_timestamps.min(),
                "latest_timestamp": unique_timestamps.max(),
                "median_interval_hours": median_interval_hours
            })


coverage_df = pd.DataFrame(coverage_records)

coverage_summary = (
    coverage_df
    .groupby("sensor")
    .agg(
        participants_with_files=("participant", "nunique"),
        total_rows=("total_rows", "sum"),
        total_unique_timestamps=("unique_timestamps", "sum"),
        earliest_timestamp=("earliest_timestamp", "min"),
        latest_timestamp=("latest_timestamp", "max"),
        median_interval_hours=("median_interval_hours", "median"),
        min_participant_interval_hours=("median_interval_hours", "min"),
        max_participant_interval_hours=("median_interval_hours", "max")
    )
    .reset_index()
)

display(coverage_summary)



### Timestamp Coverage and Sampling Frequency Audit

**Objective:** Examine the temporal coverage and typical sampling frequency of each sensor stream before integrating the data.

**Method:**
- Parsed sensor timestamps as UTC.
- Removed duplicate timestamps for interval calculations only; the original CSV files were not modified.
- Calculated total rows and unique timestamps for each sensor.
- Identified the earliest and latest timestamps.
- Calculated the median interval between consecutive unique timestamps for each participant.
- Summarized participant-level median intervals across each sensor.

**Observed results:**

| Sensor(s) | Participants with files | Median interval |
|---|---:|---:|
| BRI, MI, SHRI, SQI | 74 each | 24 hours |
| EI, SII | 74 each | 10 hours |
| PEI | 74 | 24 hours |
| PRVI | 65 | 24.09 hours |

Additional observations:
- BRI and SHRI have identical row counts and timestamp coverage statistics.
- EI and SII also have identical row counts and unique timestamp counts.
- PEI and MI begin later than BRI and SHRI.
- PRVI has substantial variation in participant-level median intervals, ranging from approximately 9.87 to 393.88 hours.
- PRVI contains 2,917 rows but only 2,722 unique timestamps, consistent with the duplicate timestamp audit.

**Interpretation:**
- Sensor streams have different typical sampling frequencies.
- A single row-wise merge would not reliably align measurements across sensors.
- PRVI requires additional investigation because its temporal coverage varies considerably between participants.
- Timestamp-based alignment and carefully defined aggregation windows will be needed during integration.

**Important limitation:** A median interval describes typical spacing, not the full distribution of gaps. Large gaps may exist even when the median interval is close to 24 hours.

**Cleaning decision:** No records were removed, resampled, or modified during this audit.


In [ ]:

prvi_gap_records = []

for country in ["GERMANY", "SPAIN"]:

    country_path = dataset_path / "index" / country

    for participant_folder in country_path.iterdir():

        if not participant_folder.is_dir():
            continue

        file_path = participant_folder / "PRVI.csv"

        if not file_path.exists():
            continue

        df = pd.read_csv(file_path)

        timestamps = (
            pd.to_datetime(df["Timestamp"], utc=True, errors="coerce")
            .dropna()
            .drop_duplicates()
            .sort_values()
        )

        intervals = timestamps.diff().dropna()

        if intervals.empty:
            continue

        gap_hours = intervals.dt.total_seconds() / 3600

        prvi_gap_records.append({
            "participant": participant_folder.name,
            "unique_timestamps": len(timestamps),
            "monitoring_span_days": (
                timestamps.max() - timestamps.min()
            ).total_seconds() / 86400,
            "median_gap_hours": gap_hours.median(),
            "maximum_gap_hours": gap_hours.max(),
            "gaps_over_48h": (gap_hours > 48).sum(),
            "gaps_over_7days": (gap_hours > 168).sum(),
            "gaps_over_14days": (gap_hours > 336).sum()
        })


prvi_gap_df = pd.DataFrame(prvi_gap_records)

display(
    prvi_gap_df
    .sort_values("maximum_gap_hours", ascending=False)
    .head(15)
    .round(2)
)



### PRVI Temporal Gap Investigation

**Objective:** Investigate irregular PRVI sampling by examining the time gaps between consecutive unique timestamps for each participant.

**Method:**
- Parsed PRVI timestamps as UTC.
- Removed duplicate timestamps for interval calculations only.
- Sorted timestamps chronologically for each participant.
- Calculated the median and maximum interval between consecutive unique timestamps.
- Counted intervals exceeding 48 hours, 7 days, and 14 days.
- Calculated each participant's monitoring span from their earliest to latest available PRVI timestamp.

**Observed results:**
- Several participants have median intervals close to 24 hours but extremely large maximum gaps.
- GER-009 has a median interval of 24 hours and a maximum gap of approximately 5,291 hours (220 days).
- GER-010 has a maximum gap of approximately 4,699 hours (196 days).
- SPA-031 has a maximum gap of approximately 3,645 hours (152 days).
- GER-025 has only 4 unique timestamps across approximately 148 days, with a median interval of approximately 394 hours.
- Other participants also show substantial interruptions in PRVI observations.

**Interpretation:**
- A near-daily median interval does not guarantee continuous daily monitoring.
- PRVI availability varies substantially between participants.
- Long gaps may affect the reliability of rolling features, personal baselines, and future risk predictions.
- The cause of these gaps cannot be established from timestamps alone.

**Cleaning decision:** No observations were removed or imputed. PRVI gaps will be considered when defining feature availability and minimum data-coverage requirements.


In [ ]:

largest_prvi_gaps = []

for country in ["GERMANY", "SPAIN"]:

    country_path = dataset_path / "index" / country

    for participant_folder in country_path.iterdir():

        if not participant_folder.is_dir():
            continue

        file_path = participant_folder / "PRVI.csv"

        if not file_path.exists():
            continue

        df = pd.read_csv(file_path)

        timestamps = (
            pd.to_datetime(df["Timestamp"], utc=True, errors="coerce")
            .dropna()
            .drop_duplicates()
            .sort_values()
            .reset_index(drop=True)
        )

        for i in range(1, len(timestamps)):

            gap_hours = (
                timestamps.iloc[i] - timestamps.iloc[i - 1]
            ).total_seconds() / 3600

            if gap_hours > 48:

                largest_prvi_gaps.append({
                    "participant": participant_folder.name,
                    "gap_start": timestamps.iloc[i - 1],
                    "gap_end": timestamps.iloc[i],
                    "gap_hours": gap_hours,
                    "gap_days": gap_hours / 24
                })


largest_prvi_gaps_df = pd.DataFrame(largest_prvi_gaps)

display(
    largest_prvi_gaps_df
    .sort_values("gap_hours", ascending=False)
    .head(20)
    .round(2)
)



### Investigation of the Largest PRVI Monitoring Gaps

**Objective:** Identify the exact time periods associated with the largest interruptions in PRVI observations.

**Method:**
- Parsed PRVI timestamps as UTC.
- Removed duplicate timestamps for interval calculations only.
- Sorted unique timestamps chronologically for each participant.
- Calculated the time difference between consecutive timestamps.
- Identified gaps exceeding 48 hours and ranked them by duration.

**Observed results:**
- GER-009 had the largest identified gap: approximately 220.48 days, from 2 September 2024 to 11 April 2025.
- GER-010 had a gap of approximately 195.79 days, from 1 October 2024 to 15 April 2025.
- SPA-031 had a gap of approximately 151.87 days, from 13 December 2024 to 14 May 2025.
- GER-025 had a gap of approximately 122 days, from 29 December 2024 to 30 April 2025.
- Several other participants also had gaps lasting multiple weeks.

**Interpretation:**
- PRVI observations can be separated by very long periods without recorded measurements.
- Some participants have near-daily median intervals despite having individual gaps lasting several months.
- Timestamp information alone cannot establish why these interruptions occurred.

**Cleaning decision:** No observations were deleted, modified, or imputed. These gaps will be considered when defining PRVI coverage requirements and time-dependent features.


In [ ]:

all_sensor_gap_records = []

for country in ["GERMANY", "SPAIN"]:

    country_path = dataset_path / "index" / country

    for participant_folder in country_path.iterdir():

        if not participant_folder.is_dir():
            continue

        participant_id = participant_folder.name

        for sensor in sensor_names:

            file_path = participant_folder / f"{sensor}.csv"

            if not file_path.exists():
                continue

            df = pd.read_csv(file_path)

            timestamps = (
                pd.to_datetime(
                    df["Timestamp"],
                    utc=True,
                    errors="coerce"
                )
                .dropna()
                .drop_duplicates()
                .sort_values()
            )

            intervals = timestamps.diff().dropna()
            gap_hours = intervals.dt.total_seconds() / 3600

            for gap in gap_hours:

                all_sensor_gap_records.append({
                    "participant": participant_id,
                    "sensor": sensor,
                    "gap_hours": gap,
                    "gap_days": gap / 24
                })


all_sensor_gaps_df = pd.DataFrame(all_sensor_gap_records)

all_sensor_gap_summary = (
    all_sensor_gaps_df
    .groupby("sensor")
    .agg(
        total_intervals=("gap_hours", "count"),
        gaps_over_48h=("gap_hours", lambda x: (x > 48).sum()),
        gaps_over_7days=("gap_hours", lambda x: (x > 168).sum()),
        participants_with_gaps_over_48h=(
            "participant",
            lambda x: x[
                all_sensor_gaps_df.loc[x.index, "gap_hours"] > 48
            ].nunique()
        ),
        maximum_gap_hours=("gap_hours", "max"),
        maximum_gap_days=("gap_days", "max")
    )
    .reset_index()
)

display(all_sensor_gap_summary.round(2))


In [ ]:

clinical_records = []

for country in ["GERMANY", "SPAIN"]:

    country_path = dataset_path / "index" / country

    for participant_folder in country_path.iterdir():

        if not participant_folder.is_dir():
            continue

        file_path = participant_folder / "clinical_indices.csv"

        if not file_path.exists():
            continue

        df = pd.read_csv(file_path)

        df["Participant_Folder"] = participant_folder.name

        clinical_records.append(df)


clinical_df = pd.concat(
    clinical_records,
    ignore_index=True
)

print("CLINICAL DATASET OVERVIEW")
print("------------------------")
print("Total records:", len(clinical_df))
print("Total participants:", clinical_df["Participant_Folder"].nunique())

print("\nTIMEPOINT DISTRIBUTION")
display(
    clinical_df["Timepoint"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nRECORDS PER PARTICIPANT")
display(
    clinical_df.groupby("Participant_Folder")
    .size()
    .value_counts()
    .sort_index()
)



### Clinical Data Structural Audit

**Objective:** Verify that clinical records are structurally consistent across all participants and expected assessment timepoints.

**Method:**
- Loaded `clinical_indices.csv` from all participant folders.
- Combined the records into a temporary DataFrame for analysis.
- Counted total records and unique participants.
- Examined the distribution of clinical assessment timepoints.
- Counted the number of records available for each participant.

**Observed results:**
- Total participants: 74.
- Total clinical records: 370.
- Each of the five expected timepoints (T1–T5) occurs exactly 74 times.
- All 74 participants have exactly five clinical records.

**Interpretation:**
- The clinical dataset has a consistent row structure.
- Every participant has a record corresponding to each expected assessment timepoint.
- However, the presence of a record does not guarantee that its clinical measurements are populated.

**Cleaning decision:** No clinical records were removed or modified.


In [ ]:

# Exclude the temporary participant identifier added during ingestion
clinical_columns = [
    col for col in clinical_df.columns
    if col != "Participant_Folder"
]

clinical_missing_records = []

for column in clinical_columns:

    missing_count = clinical_df[column].isna().sum()

    if missing_count > 0:

        participants_affected = (
            clinical_df.loc[
                clinical_df[column].isna(),
                "Participant_Folder"
            ].nunique()
        )

        clinical_missing_records.append({
            "column": column,
            "total_values": len(clinical_df),
            "missing_values": missing_count,
            "participants_affected": participants_affected,
            "missing_percentage": (
                missing_count / len(clinical_df)
            ) * 100
        })


clinical_missing_summary = pd.DataFrame(
    clinical_missing_records
)

clinical_missing_summary = (
    clinical_missing_summary
    .sort_values("missing_percentage", ascending=False)
    .reset_index(drop=True)
)

print("CLINICAL MISSING-VALUE AUDIT")
print("----------------------------")

if clinical_missing_summary.empty:
    print("No missing values found.")
else:
    display(clinical_missing_summary.round(2))



### Clinical Missing-Value Audit

**Objective:** Quantify missing clinical information across all participants and identify variables with substantial missingness.

**Method:**
- Examined missing values across all 370 clinical records.
- Calculated missing-value counts and percentages for each column.
- Counted the number of participants affected by missing values in each column.

**Observed results:**

- DLCO and DLCO_pred have 83.24% missingness.
- FEV1, FVC, their predicted percentages, and Lung Ratio have approximately 68.92% missingness.
- Several clinical measurements, including CAT, CCQ, blood pressure, heart rate and EQ-5D variables, have approximately 40.81% missingness.
- Demographic fields such as Sex, Year of Birth, Height and Weight have approximately 39.46% missingness.
- Missing values occur across all 74 participants for several clinical variables.

**README facts:**
- Clinical assessments are organized into five timepoints, T1–T5.
- Empty clinical cells may indicate that a measurement was not collected or was not applicable.

**Interpretation:**
- Clinical missingness is substantial and may be related to the assessment schedule.
- Similar missing-value counts across multiple columns may indicate that groups of measurements are missing together.
- The missingness pattern needs to be investigated by timepoint before selecting an imputation or feature-exclusion strategy.

**Cleaning decision:** No missing values were imputed, and no clinical columns or records were removed.


In [ ]:

# Exclude identifiers and the timepoint column from measurement analysis
excluded_columns = [
    "Patient-ID",
    "Timepoint",
    "Participant_Folder"
]

clinical_measurement_columns = [
    col for col in clinical_df.columns
    if col not in excluded_columns
]

# Calculate missing percentage separately for each timepoint
missing_by_timepoint = (
    clinical_df
    .groupby("Timepoint")[clinical_measurement_columns]
    .apply(lambda group: group.isna().mean() * 100)
    .T
)

print("MISSING VALUE PERCENTAGE BY TIMEPOINT")
print("-------------------------------------")

display(missing_by_timepoint.round(2))

# Also calculate how many completely empty clinical records exist
measurement_columns = [
    col for col in clinical_measurement_columns
    if col != "Visit Date"
]

clinical_df["all_measurements_missing"] = (
    clinical_df[measurement_columns]
    .isna()
    .all(axis=1)
)

print("\nCOMPLETELY EMPTY MEASUREMENT RECORDS BY TIMEPOINT")

display(
    clinical_df
    .groupby("Timepoint")["all_measurements_missing"]
    .agg(
        total_records="count",
        completely_empty_records="sum"
    )
)



### Clinical Missingness by Timepoint

**Objective:** Investigate whether clinical missingness is distributed randomly or concentrated at particular assessment timepoints.

**Method:**
- Calculated the percentage of missing values for each clinical variable separately at T1–T5.
- Counted records with all clinical measurement fields missing at each timepoint.
- Excluded identifiers and the timepoint column from the measurement completeness calculation.

**Observed results:**

| Timepoint | Completely empty records | Records with at least one measurement |
|---|---:|---:|
| T1 | 0 | 74 |
| T2 | 9 | 65 |
| T3 | 25 | 49 |
| T4 | 50 | 24 |
| T5 | 62 | 12 |

Additional observations:
- General clinical measurements show increasing missingness at later timepoints.
- FEV1, FVC and Lung Ratio are 100% missing at T2 and T4.
- DLCO and DLCO_pred are 100% missing at T2–T5.
- Six-Minute Walk Distance is 100% missing at T2 and T4.
- By T5, only 12 of 74 records contain at least one clinical measurement.

**Interpretation:**
- Clinical data availability decreases substantially across later assessment timepoints.
- Missingness is structured by timepoint and measurement type rather than appearing uniformly distributed.
- Clinical variables must be handled according to their actual availability.
- The reason for the missingness cannot be established from this audit alone.

**Cleaning decision:** No records or columns were removed, and no missing values were imputed.


In [ ]:

# Check duplicate participant-timepoint combinations
duplicate_visits = clinical_df[
    clinical_df.duplicated(
        subset=["Participant_Folder", "Timepoint"],
        keep=False
    )
].sort_values(["Participant_Folder", "Timepoint"])

# Check whether internal Patient-ID maps consistently to folders
id_mapping = (
    clinical_df[
        ["Participant_Folder", "Patient-ID"]
    ]
    .drop_duplicates()
)

folders_per_patient_id = (
    id_mapping.groupby("Patient-ID")["Participant_Folder"]
    .nunique()
)

patient_ids_per_folder = (
    id_mapping.groupby("Participant_Folder")["Patient-ID"]
    .nunique()
)

# Check missing identifiers and timepoints
print("CLINICAL IDENTIFIER AND DUPLICATE AUDIT")
print("---------------------------------------")

print("Duplicate participant-timepoint rows:", len(duplicate_visits))

print(
    "Missing Patient-ID values:",
    clinical_df["Patient-ID"].isna().sum()
)

print(
    "Missing Timepoint values:",
    clinical_df["Timepoint"].isna().sum()
)

print(
    "Patient IDs linked to multiple folders:",
    (folders_per_patient_id > 1).sum()
)

print(
    "Folders linked to multiple Patient IDs:",
    (patient_ids_per_folder > 1).sum()
)

print("\nDUPLICATE VISITS")
display(duplicate_visits)

print("\nPATIENT IDs LINKED TO MULTIPLE FOLDERS")
display(folders_per_patient_id[folders_per_patient_id > 1])

print("\nFOLDERS LINKED TO MULTIPLE PATIENT IDs")
display(patient_ids_per_folder[patient_ids_per_folder > 1])



### Clinical Identifier and Duplicate Visit Audit

**Objective:** Verify participant identity consistency and detect duplicate clinical assessment records.

**Method:**
- Checked for duplicate combinations of participant folder and timepoint.
- Checked for missing Patient-ID and Timepoint values.
- Verified that each Patient-ID maps to only one participant folder.
- Verified that each participant folder maps to only one Patient-ID.

**Observed results:**
- Duplicate participant-timepoint records: 0.
- Missing Patient-ID values: 0.
- Missing Timepoint values: 0.
- Patient IDs linked to multiple folders: 0.
- Participant folders linked to multiple Patient IDs: 0.

**Interpretation:**
- Clinical participant identifiers are internally consistent.
- No duplicate assessment records were detected.
- Clinical records can be associated with participant folders without observed ID mapping conflicts.

**Cleaning decision:** No records or identifiers were modified.


In [ ]:

exacerbation_records = []

for country in ["GERMANY", "SPAIN"]:

    country_path = dataset_path / "index" / country

    for participant_folder in country_path.iterdir():

        if not participant_folder.is_dir():
            continue

        file_path = participant_folder / "exacerbations.csv"

        if not file_path.exists():
            continue

        df = pd.read_csv(file_path)

        df["Participant_Folder"] = participant_folder.name

        exacerbation_records.append(df)


if exacerbation_records:

    exacerbation_df = pd.concat(
        exacerbation_records,
        ignore_index=True
    )

    print("EXACERBATION DATASET OVERVIEW")
    print("-----------------------------")

    print("Total exacerbation records:", len(exacerbation_df))

    print(
        "Participants with recorded events:",
        exacerbation_df["Participant_Folder"].nunique()
    )

    print("\nEVENT TYPE DISTRIBUTION")

    display(
        exacerbation_df["Type"]
        .value_counts(dropna=False)
    )

    print("\nEVENTS PER PARTICIPANT")

    display(
        exacerbation_df.groupby("Participant_Folder")
        .size()
        .describe()
    )

    print("\nFIRST FIVE RECORDS")

    display(exacerbation_df.head())

else:
    print("No exacerbation records found.")



### Exacerbation Event Inventory

**Objective:** Understand the number, severity, and participant distribution of recorded exacerbation events.

**Observed results:**
- Total exacerbation records: 39.
- Participants with recorded events: 24 out of 74.
- Moderate events: 29 (74.36%).
- Severe events: 10 (25.64%).
- Average events per participant with recorded events: 1.625.
- Minimum events per affected participant: 1.
- Maximum events per affected participant: 4.

**Interpretation:**
- The dataset contains both moderate and severe exacerbation events.
- Some participants have multiple recorded events, so event records are not independent participant observations.
- The relatively small number of events is an important limitation for future predictive modelling.

**Dataset README fact:** Participants without an `exacerbations.csv` file are described as having no exacerbation events recorded during the monitoring period. This should not be interpreted as proof that they never experienced an exacerbation.

**Cleaning decision:** No event records were modified or removed.


In [ ]:

# Exacerbation event date and identifier validation

event_audit = exacerbation_df.copy()

# Parse dates without changing the original dataset
event_audit["Parsed_Date"] = pd.to_datetime(
    event_audit["Date"],
    errors="coerce"
)

# Check for duplicate events:
# same participant, date, and event type
duplicate_events = event_audit[
    event_audit.duplicated(
        subset=["Participant_Folder", "Date", "Type"],
        keep=False
    )
].sort_values(["Participant_Folder", "Date", "Type"])

# Check whether Kit-id matches the participant folder
kit_folder_mismatches = event_audit[
    event_audit["Kit-id"] != event_audit["Participant_Folder"]
]

# Check PatientId-to-folder mapping consistency
patient_folder_mapping = (
    event_audit[["PatientId", "Participant_Folder"]]
    .drop_duplicates()
)

folders_per_patient_id = (
    patient_folder_mapping.groupby("PatientId")["Participant_Folder"]
    .nunique()
)

patient_ids_per_folder = (
    patient_folder_mapping.groupby("Participant_Folder")["PatientId"]
    .nunique()
)

print("EXACERBATION DATE AND IDENTIFIER AUDIT")
print("--------------------------------------")

print("Missing dates:", event_audit["Date"].isna().sum())
print("Unparseable dates:", event_audit["Parsed_Date"].isna().sum())

print(
    "Missing PatientId:",
    event_audit["PatientId"].isna().sum()
)

print(
    "Missing Kit-id:",
    event_audit["Kit-id"].isna().sum()
)

print(
    "Missing event Type:",
    event_audit["Type"].isna().sum()
)

print("Duplicate event records:", len(duplicate_events))
print("Kit-id / folder mismatches:", len(kit_folder_mismatches))

print(
    "Patient IDs linked to multiple folders:",
    (folders_per_patient_id > 1).sum()
)

print(
    "Folders linked to multiple Patient IDs:",
    (patient_ids_per_folder > 1).sum()
)

print("\nEVENT DATE RANGE")

print("Earliest event:", event_audit["Parsed_Date"].min())
print("Latest event:", event_audit["Parsed_Date"].max())

print("\nUNEXPECTED EVENT TYPES")

display(
    event_audit[
        ~event_audit["Type"].isin(["Moderate", "Severe"])
    ]["Type"].value_counts(dropna=False)
)

print("\nDUPLICATE EVENT RECORDS")

display(duplicate_events)

print("\nKIT-ID / FOLDER MISMATCHES")

display(kit_folder_mismatches)

print("\nPATIENT ID MAPPING ISSUES")

display(folders_per_patient_id[folders_per_patient_id > 1])
display(patient_ids_per_folder[patient_ids_per_folder > 1])



### Exacerbation Date and Identifier Validation

**Objective:** Check the completeness and consistency of exacerbation event dates, participant identifiers, and event classifications.

**Observed results:**
- Missing event dates: 0.
- Unparseable event dates: 0.
- Missing PatientId, Kit-id, and Type values: 0.
- Duplicate event records: 0.
- Kit-id / participant-folder mismatches: 0.
- Patient IDs linked to multiple folders: 0.
- Participant folders linked to multiple Patient IDs: 0.
- Unexpected event types: 0.
- Earliest event: 2024-08-19.
- Latest event: 2025-06-30.

**Interpretation:**
- The event records have consistent identifiers and valid date formats.
- No duplicate or structurally inconsistent event records were detected.
- The latest event date extends beyond the latest sensor observation date found in the dataset-wide timestamp audit (2025-05-31).

**Important consideration:** The event date range and sensor observation range do not fully overlap. Participant-level sensor coverage must be checked before constructing prediction targets.

**Cleaning decision:** No event records were modified or removed.


In [ ]:

sensor_names = [
    "BRI", "EI", "MI", "PEI",
    "PRVI", "SHRI", "SII", "SQI"
]

# Get the earliest and latest timestamp for every
# participant and sensor stream.
coverage_records = []

for country in ["GERMANY", "SPAIN"]:

    country_path = dataset_path / "index" / country

    for participant_folder in country_path.iterdir():

        if not participant_folder.is_dir():
            continue

        for sensor in sensor_names:

            file_path = participant_folder / f"{sensor}.csv"

            if not file_path.exists():
                continue

            df = pd.read_csv(file_path)

            timestamps = pd.to_datetime(
                df["Timestamp"],
                errors="coerce",
                utc=True
            ).dropna()

            if len(timestamps) == 0:
                continue

            coverage_records.append({
                "Participant_Folder": participant_folder.name,
                "Sensor": sensor,
                "First_Observation": timestamps.min(),
                "Last_Observation": timestamps.max(),
                "Total_Rows": len(df)
            })


coverage_df = pd.DataFrame(coverage_records)

# Compare each event against its participant's sensor coverage.
event_coverage = event_audit.copy()

event_coverage["Event_Date_UTC"] = pd.to_datetime(
    event_coverage["Date"],
    utc=True
)

event_coverage = event_coverage.merge(
    coverage_df,
    on="Participant_Folder",
    how="left"
)

event_coverage["Event_After_Last_Observation"] = (
    event_coverage["Event_Date_UTC"] >
    event_coverage["Last_Observation"]
)

event_coverage["Days_After_Last_Observation"] = (
    event_coverage["Event_Date_UTC"] -
    event_coverage["Last_Observation"]
).dt.total_seconds() / 86400

# Summary for each sensor
print("EVENTS OCCURRING AFTER SENSOR COVERAGE")
print("--------------------------------------")

display(
    event_coverage.groupby("Sensor").agg(
        Total_Event_Comparisons=("Date", "count"),
        Events_After_Last_Observation=(
            "Event_After_Last_Observation", "sum"
        ),
        Participants_Compared=(
            "Participant_Folder", "nunique"
        )
    )
)

print("\nEVENTS AFTER ALL AVAILABLE SENSOR STREAMS")

# For each event, determine the latest observation
# across all available sensors for that participant.
latest_by_participant = (
    coverage_df.groupby("Participant_Folder")["Last_Observation"]
    .max()
    .rename("Latest_Any_Sensor_Observation")
)

event_coverage_summary = event_audit.merge(
    latest_by_participant,
    on="Participant_Folder",
    how="left"
)

event_coverage_summary["Days_After_All_Sensors"] = (
    pd.to_datetime(
        event_coverage_summary["Date"],
        utc=True
    ) -
    event_coverage_summary["Latest_Any_Sensor_Observation"]
).dt.total_seconds() / 86400

events_after_all = event_coverage_summary[
    event_coverage_summary["Days_After_All_Sensors"] > 0
]

display(
    events_after_all[
        [
            "Participant_Folder",
            "Date",
            "Type",
            "Latest_Any_Sensor_Observation",
            "Days_After_All_Sensors"
        ]
    ].sort_values("Date")
)



### Exacerbation Events vs. Sensor Monitoring Coverage

**Objective:** Determine whether participants had sensor observations available at the time of their recorded exacerbation events.

**Method:**
- Compared each exacerbation date with the last available timestamp for each sensor stream belonging to that participant.
- Also compared each event against the latest observation across all available sensor streams for that participant.
- Did not modify or remove any event records.

**Observed results:**
- Seven sensor streams (BRI, EI, MI, PEI, SHRI, SII, SQI) each had 10 events occurring after their participant's last observation.
- PRVI had 37 event comparisons, of which 28 occurred after the last PRVI observation.
- 10 events occurred after the latest observation across all available sensor streams.
- These 10 events affected 9 participants, because SPA-001 had two such events.
- The delay between the last sensor observation and these events ranged from 2.5 to 179.5 days.

**Interpretation:**
- Some exacerbation events do not have sensor observations available near their event dates.
- PRVI has particularly limited temporal coverage.
- Events occurring after sensor monitoring ended cannot automatically be treated as examples with recent pre-event sensor measurements.
- The event records should be preserved. Their suitability for model training must be evaluated using the required historical observation windows.

**Cleaning decision:** No sensor observations or exacerbation events were deleted or modified.


In [ ]:

# Count sensor observations in the historical windows
# before each exacerbation event.

lookback_days = [7, 14, 30]

event_window_records = []

for _, event in event_audit.iterrows():

    participant = event["Participant_Folder"]

    event_date = pd.to_datetime(
        event["Date"],
        utc=True
    )

    country = "GERMANY" if participant.startswith("GER") else "SPAIN"

    participant_path = (
        dataset_path / "index" / country / participant
    )

    for sensor in sensor_names:

        file_path = participant_path / f"{sensor}.csv"

        if not file_path.exists():
            continue

        df = pd.read_csv(file_path)

        df["Timestamp"] = pd.to_datetime(
            df["Timestamp"],
            errors="coerce",
            utc=True
        )

        for days in lookback_days:

            window_start = event_date - pd.Timedelta(days=days)

            # Only observations strictly before the event date
            window_df = df[
                (df["Timestamp"] >= window_start) &
                (df["Timestamp"] < event_date)
            ]

            valid_count = 0

            if "index_validity" in window_df.columns:
                valid_count = (
                    window_df["index_validity"] == 1
                ).sum()

            event_window_records.append({
                "Participant_Folder": participant,
                "Event_Date": event["Date"],
                "Event_Type": event["Type"],
                "Sensor": sensor,
                "Lookback_Days": days,
                "Total_Observations": len(window_df),
                "Valid_Observations": valid_count
            })


event_window_df = pd.DataFrame(event_window_records)

# Summarize the number of events with at least one valid observation
# in each sensor and lookback window.

window_summary = (
    event_window_df
    .groupby(["Sensor", "Lookback_Days"])
    .agg(
        Events_Compared=("Event_Date", "count"),
        Events_With_Any_Observation=(
            "Total_Observations",
            lambda x: (x > 0).sum()
        ),
        Events_With_Valid_Observation=(
            "Valid_Observations",
            lambda x: (x > 0).sum()
        ),
        Total_Observations=("Total_Observations", "sum"),
        Total_Valid_Observations=("Valid_Observations", "sum")
    )
    .reset_index()
)

print("PRE-EVENT SENSOR COVERAGE SUMMARY")
print("---------------------------------")

display(
    window_summary.sort_values(
        ["Sensor", "Lookback_Days"]
    )
)



### Pre-Event Sensor Coverage Audit

**Objective:** Evaluate sensor data availability in the 7-day, 14-day, and 30-day periods preceding recorded exacerbation events.

**Method:**
- Compared each event date with the participant's sensor observations.
- Examined three historical windows: 7, 14, and 30 days.
- Excluded observations on the event date because event records contain dates but not exact event times.
- Counted total observations and observations with `index_validity = 1`.

**Observed results:**

In the 7-day window:
- BRI: 28 events had observations; 3 had at least one valid observation.
- EI: 28 had observations; 8 had at least one valid observation.
- MI: 28 had observations; 13 had at least one valid observation.
- PEI: 28 had observations; 15 had at least one valid observation.
- PRVI: 7 of 37 compared events had observations, and all 7 had at least one valid observation.
- SHRI: 28 had observations; 3 had at least one valid observation.
- SII: 28 had observations; 8 had at least one valid observation.
- SQI: 28 had observations; 3 had at least one valid observation.

**Interpretation:**
- Sensor availability near exacerbation events is limited.
- Some sensor streams contain timestamped observations that are marked as invalid health indices.
- Extending the historical window generally increases the number of events with valid observations.
- PRVI has particularly limited coverage near events.

**Limitations:**
- Duplicate timestamp rows were counted separately in the original observation totals.
- Having at least one valid observation does not guarantee sufficient coverage for reliable feature calculation.
- These results describe positive-event windows only; they do not establish the availability of suitable non-event windows.

**Cleaning decision:** No records were removed, deduplicated, or imputed.


In [ ]:

unique_window_records = []

for _, event in event_audit.iterrows():

    participant = event["Participant_Folder"]

    event_date = pd.to_datetime(
        event["Date"],
        utc=True
    )

    country = "GERMANY" if participant.startswith("GER") else "SPAIN"

    participant_path = (
        dataset_path / "index" / country / participant
    )

    for sensor in sensor_names:

        file_path = participant_path / f"{sensor}.csv"

        if not file_path.exists():
            continue

        df = pd.read_csv(file_path)

        df["Timestamp"] = pd.to_datetime(
            df["Timestamp"],
            errors="coerce",
            utc=True
        )

        for days in lookback_days:

            window_start = event_date - pd.Timedelta(days=days)

            window_df = df[
                (df["Timestamp"] >= window_start) &
                (df["Timestamp"] < event_date)
            ].copy()

            # Keep only rows with a valid calculated health index
            valid_df = window_df[
                window_df["index_validity"] == 1
            ]

            # Count unique timestamps, not duplicate rows
            unique_valid_timestamps = (
                valid_df["Timestamp"].nunique()
            )

            unique_window_records.append({
                "Participant_Folder": participant,
                "Event_Date": event["Date"],
                "Event_Type": event["Type"],
                "Sensor": sensor,
                "Lookback_Days": days,
                "Unique_Valid_Timestamps": unique_valid_timestamps
            })


unique_window_df = pd.DataFrame(unique_window_records)

unique_coverage_summary = (
    unique_window_df
    .groupby(["Sensor", "Lookback_Days"])
    .agg(
        Events_Compared=("Event_Date", "count"),
        Events_With_Valid_Data=(
            "Unique_Valid_Timestamps",
            lambda x: (x > 0).sum()
        ),
        Median_Valid_Timestamps=(
            "Unique_Valid_Timestamps", "median"
        ),
        Mean_Valid_Timestamps=(
            "Unique_Valid_Timestamps", "mean"
        ),
        Maximum_Valid_Timestamps=(
            "Unique_Valid_Timestamps", "max"
        )
    )
    .reset_index()
)

print("UNIQUE VALID TIMESTAMP COVERAGE")
print("--------------------------------")

display(
    unique_coverage_summary.sort_values(
        ["Sensor", "Lookback_Days"]
    ).round(2)
)


In [ ]:

# Count unique timestamps where BOTH validity flags are 1
# within the 7, 14, and 30-day pre-event windows.

joint_validity_records = []

for _, event in event_audit.iterrows():

    participant = event["Participant_Folder"]

    event_date = pd.to_datetime(
        event["Date"],
        utc=True
    )

    country = "GERMANY" if participant.startswith("GER") else "SPAIN"

    participant_path = (
        dataset_path / "index" / country / participant
    )

    for sensor in sensor_names:

        file_path = participant_path / f"{sensor}.csv"

        if not file_path.exists():
            continue

        df = pd.read_csv(file_path)

        df["Timestamp"] = pd.to_datetime(
            df["Timestamp"],
            errors="coerce",
            utc=True
        )

        for days in [7, 14, 30]:

            window_start = event_date - pd.Timedelta(days=days)

            window_df = df[
                (df["Timestamp"] >= window_start) &
                (df["Timestamp"] < event_date)
            ].copy()

            # Require both flags to be valid
            jointly_valid = window_df[
                (window_df["data_validity"] == 1) &
                (window_df["index_validity"] == 1)
            ]

            # Count unique timestamps, not duplicate rows
            unique_valid_timestamps = (
                jointly_valid["Timestamp"].dropna().nunique()
            )

            joint_validity_records.append({
                "Participant_Folder": participant,
                "Event_Date": event["Date"],
                "Event_Type": event["Type"],
                "Sensor": sensor,
                "Lookback_Days": days,
                "Jointly_Valid_Timestamps": unique_valid_timestamps
            })


joint_validity_df = pd.DataFrame(joint_validity_records)

joint_validity_summary = (
    joint_validity_df
    .groupby(["Sensor", "Lookback_Days"])
    .agg(
        Events_Compared=("Event_Date", "count"),
        Events_With_Jointly_Valid_Data=(
            "Jointly_Valid_Timestamps",
            lambda x: (x > 0).sum()
        ),
        Median_Valid_Timestamps=(
            "Jointly_Valid_Timestamps", "median"
        ),
        Mean_Valid_Timestamps=(
            "Jointly_Valid_Timestamps", "mean"
        ),
        Maximum_Valid_Timestamps=(
            "Jointly_Valid_Timestamps", "max"
        )
    )
    .reset_index()
)

print("JOINT VALIDITY: UNIQUE PRE-EVENT TIMESTAMPS")
print("-------------------------------------------")

display(joint_validity_summary)



### Joint Validity Audit of Pre-Event Sensor Observations

**Objective:** Verify sensor coverage before exacerbation events using both dataset validity flags.

**Method:**
- Examined 7-day, 14-day, and 30-day windows before each event.
- Counted unique timestamps satisfying both `data_validity == 1` and `index_validity == 1`.
- Excluded observations on the event date because exact event times were unavailable.

**Observed results:**
- Joint-validity counts were identical to the previous unique valid timestamp audit.
- Within 30 days, MI had the highest coverage: 24 of 39 events had at least one jointly valid timestamp.
- PEI had 22 of 39 events with at least one jointly valid timestamp.
- PRVI had 11 of 37 event comparisons with at least one jointly valid timestamp.
- BRI and SHRI each had 5 of 39.
- SQI had 4 of 39.

**Interpretation:**
- Requiring both validity flags did not further reduce the number of valid observations.
- This agrees with the earlier dataset-wide finding that no rows had `data_validity == 0` while `index_validity == 1`.
- Sensor coverage before exacerbations is limited, even with a 30-day lookback.
- At least one valid observation is not sufficient evidence of adequate prediction-window coverage.

**Cleaning decision:** No observations or events were modified or removed.


In [ ]:

# Investigate all-zero measurement rows marked as valid

for sensor in ["EI", "SII"]:

    matching_rows = []

    for country in ["GERMANY", "SPAIN"]:

        country_path = dataset_path / "index" / country

        for participant_folder in country_path.iterdir():

            if not participant_folder.is_dir():
                continue

            file_path = participant_folder / f"{sensor}.csv"

            if not file_path.exists():
                continue

            df = pd.read_csv(file_path)

            measurement_columns = [
                col for col in df.columns
                if col not in [
                    "Timestamp",
                    "index_validity",
                    "data_validity"
                ]
            ]

            all_zero = (
                df[measurement_columns]
                .fillna(0)
                .eq(0)
                .all(axis=1)
            )

            suspicious_rows = df[
                all_zero & (df["index_validity"] == 1)
            ].copy()

            if not suspicious_rows.empty:

                suspicious_rows["Participant_Folder"] = (
                    participant_folder.name
                )

                matching_rows.append(suspicious_rows)

    print(f"\n{sensor}: ALL-ZERO ROWS MARKED VALID")

    if matching_rows:
        result = pd.concat(matching_rows, ignore_index=True)

        print("Number of rows:", len(result))
        print(
            "Participants affected:",
            result["Participant_Folder"].nunique()
        )

        display(result.head(20))

    else:
        print("No matching rows found.")



### Investigation of All-Zero EI and SII Records Marked Valid

**Objective:** Investigate sensor records where every measurement is zero despite both validity flags indicating valid data.

**Observed results:**

| Sensor | All-zero rows marked valid | Participants affected |
|---|---:|---:|
| EI | 13 | 5 |
| SII | 10 | 3 |
| **Total** | **23** | **5 unique participants** |

All identified rows had:
- `index_validity = 1`
- `data_validity = 1`
- All measurement columns equal to zero.

**Additional observation:**
- EI and SII anomalies occur at overlapping timestamps for GER-005, GER-030, and SPA-015.
- SPA-015 accounts for several of the repeated anomalous observations.

**Interpretation:**
- These records are unusual because the validity flags indicate valid measurements despite every measurement being zero.
- The overlapping EI and SII timestamps may indicate a shared upstream recording or processing issue.
- The underlying cause cannot be established from the available records alone.

**Cleaning decision:**
- Flag these 23 records for further investigation.
- Do not automatically delete them or replace their values with missing values.
- Preserve the original records for traceability.


In [ ]:

# Establish preprocessing output directories

interim_path = project_path / "data" / "interim"
processed_path = project_path / "data" / "processed"

interim_path.mkdir(parents=True, exist_ok=True)
processed_path.mkdir(parents=True, exist_ok=True)

print("PREPROCESSING DIRECTORIES")
print("-------------------------")
print("Raw data:", raw_path)
print("Interim data:", interim_path)
print("Processed data:", processed_path)

print("\nRaw dataset still exists:", dataset_path.exists())
